# Comparative Analysis: Time-Series Forecasting on Financial Data

Compare **ARIMA**, **LSTM**, **Prophet**, **Transformer**, and **baselines** (moving average, last value) on the same train/test split. Metrics: RMSE, MAE, MAPE, Directional Accuracy.

## 1. Setup and load data

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

df = pd.read_csv(ROOT / "data" / "stock_data.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
print(df.head())
print(f"\nShape: {df.shape}")

## 2. Run benchmark (or load existing results)

In [ ]:
from benchmark import run_benchmark, save_metrics, plot_comparison

results, predictions, y_test, dates_test, _ = run_benchmark(
    train_ratio=0.8,
    lstm_epochs=30,
    transformer_epochs=30,
)

metrics_df = pd.DataFrame(results).T
print("Metrics:")
display(metrics_df.round(4))

## 3. Visualize all models vs actual

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(y_test))
ax.plot(x, y_test, "k-", label="Actual", linewidth=2)
for name, pred in predictions.items():
    if np.any(np.isfinite(pred)):
        ax.plot(x, pred, "--", label=name, alpha=0.8)
ax.set_xlabel("Time step")
ax.set_ylabel("Close price")
ax.set_title("All models vs actual (test set)")
ax.legend(loc="best", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Conclusion and research angle

- **ARIMA**: Theoretically grounded in stationarity; good when series (e.g. returns) is close to stationary. Often worse on raw price levels with strong trend.
- **LSTM**: Often best raw accuracy on prices; captures nonlinearity and long memory. Needs enough data and tuning.
- **Prophet**: Good for daily data with weekly/yearly seasonality; interpretable trend + seasonality.
- **Transformer**: Can match or beat LSTM with sufficient sequence length and data; more compute.
- **Baselines**: Last value and moving average set a floor; beating them is necessary but not sufficient.

**Research question:** Can we combine statistical rigor (ARIMA, interpretability) with ML flexibility (LSTM/Transformer) via hybrid or ensemble methods?